# PatchCore Review Notebook (EfficientNet-B0, x224)

This curated notebook reviews the checked-in EfficientNet-B0 PatchCore results without retraining.


## Submission Context

- Dataset notebook: `data/dataset/x224/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x224/benchmark_50k_5pct/data_config.toml`
- Metadata CSV: `data/processed/x224/wm811k/metadata_50k_5pct.csv`
- Artifact root: `experiments/anomaly_detection/patchcore/efficientnet_b0/x224/main/artifacts/patchcore_efficientnet_b0_5pct`
- Mode: results review from saved CSV artifacts
- Checkpoint status: no checked-in checkpoint for this branch


## Imports and Saved Results

This cell loads the saved summary, score CSVs, and local metadata needed for review plots and defect analysis.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.evaluation import summarize_threshold_metrics

ARTIFACT_ROOT = REPO_ROOT / "experiments/anomaly_detection/patchcore/efficientnet_b0/x224/main/artifacts/patchcore_efficientnet_b0_5pct"
RESULTS_DIR = ARTIFACT_ROOT / "results"
EVAL_DIR = RESULTS_DIR / "evaluation"
PLOTS_DIR = ARTIFACT_ROOT / "plots"
METADATA_PATH = REPO_ROOT / "data/processed/x224/wm811k/metadata_50k_5pct.csv"

metadata = pd.read_csv(METADATA_PATH)
test_metadata = metadata[metadata["split"] == "test"].reset_index(drop=True)
summary = json.loads((RESULTS_DIR / "summary.json").read_text(encoding="utf-8"))
variant_summary_df = pd.read_csv(RESULTS_DIR / "variant_summary.csv")
best_row_df = pd.read_csv(RESULTS_DIR / "best_row.csv")
val_scores_df = pd.read_csv(EVAL_DIR / "val_scores.csv")
test_scores_df = pd.read_csv(EVAL_DIR / "test_scores.csv")
threshold_sweep_df = pd.read_csv(EVAL_DIR / "threshold_sweep.csv")
threshold = float(summary["threshold"])
metrics = summarize_threshold_metrics(test_scores_df["is_anomaly"].to_numpy(), test_scores_df["score"].to_numpy(), threshold)

display(variant_summary_df)
display(best_row_df)


## Metrics and Plots

These cells recreate the main evaluation figures from the saved score CSVs and store them under `plots/`.

In [ ]:
metrics_df = pd.DataFrame(
    [
        {"metric": "precision", "value": metrics["precision"]},
        {"metric": "recall", "value": metrics["recall"]},
        {"metric": "f1", "value": metrics["f1"]},
        {"metric": "auroc", "value": metrics["auroc"]},
        {"metric": "auprc", "value": metrics["auprc"]},
        {"metric": "threshold", "value": threshold},
    ]
)
confusion_df = pd.DataFrame(metrics["confusion_matrix"], index=["true_normal", "true_anomaly"], columns=["pred_normal", "pred_anomaly"])
display(metrics_df)
display(confusion_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(val_scores_df["score"], bins=30, alpha=0.85, color="#264653")
axes[0].axvline(threshold, color="red", linestyle="--")
axes[0].set_title("Validation Normal Score Distribution")
axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 0]["score"], bins=30, alpha=0.7, label="normal", color="#577590")
axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 1]["score"], bins=30, alpha=0.7, label="anomaly", color="#e76f51")
axes[1].axvline(threshold, color="red", linestyle="--")
axes[1].set_title("Test Score Distribution")
axes[1].legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / "score_distribution.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_sweep_df.iloc[:, 0], threshold_sweep_df["precision"], label="precision")
ax.plot(threshold_sweep_df.iloc[:, 0], threshold_sweep_df["recall"], label="recall")
ax.plot(threshold_sweep_df.iloc[:, 0], threshold_sweep_df["f1"], label="f1")
ax.set_title("Threshold Sweep")
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / "threshold_sweep.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)


## Defect Analysis

This cell recomputes failure analysis from the saved test scores and local metadata, then saves the resulting tables and plot.

In [ ]:
analysis_df = test_metadata.copy()
analysis_df["score"] = test_scores_df.reset_index(drop=True)["score"]
analysis_df["predicted_anomaly"] = (analysis_df["score"] > threshold).astype(int)
analysis_df["error_type"] = "tn"
analysis_df.loc[(analysis_df["is_anomaly"] == 0) & (analysis_df["predicted_anomaly"] == 1), "error_type"] = "fp"
analysis_df.loc[(analysis_df["is_anomaly"] == 1) & (analysis_df["predicted_anomaly"] == 0), "error_type"] = "fn"
analysis_df.loc[(analysis_df["is_anomaly"] == 1) & (analysis_df["predicted_anomaly"] == 1), "error_type"] = "tp"

error_summary_df = (
    analysis_df.groupby("error_type")
    .agg(count=("error_type", "size"), mean_score=("score", "mean"))
    .reindex(["tp", "fn", "fp", "tn"])
)
defect_recall_df = (
    analysis_df[analysis_df["is_anomaly"] == 1]
    .groupby("defect_type")
    .agg(count=("defect_type", "size"), detected=("predicted_anomaly", "sum"), mean_score=("score", "mean"))
    .sort_values(["detected", "count"], ascending=[False, False])
)
defect_recall_df["recall"] = defect_recall_df["detected"] / defect_recall_df["count"]

analysis_df.to_csv(EVAL_DIR / "analysis_with_predictions.csv", index=False)
error_summary_df.reset_index().to_csv(EVAL_DIR / "error_summary.csv", index=False)
defect_recall_df.reset_index().to_csv(EVAL_DIR / "defect_recall.csv", index=False)

top_defects_df = defect_recall_df.head(10).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(error_summary_df.index.astype(str), error_summary_df["count"], color="#e76f51")
axes[0].set_title("Prediction Outcome Counts")
axes[1].barh(top_defects_df["defect_type"], top_defects_df["recall"], color="#8ab17d")
axes[1].set_xlim(0.0, 1.0)
axes[1].invert_yaxis()
axes[1].set_title("Top Defect-Type Recall")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "defect_breakdown.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

display(error_summary_df)
display(defect_recall_df)
